In [ ]:
MODEL = "qwen/qwen3.8-27b:free"
IMAGE = "input/amostra_diversa/captura/300/01-1-cinza-300/p01.png"
KINDS = ("compact",)  # entradas do llm: compact, markdown, full_json
OUTPUT = "output/notebook"
CPU_THREADS = 2
REASONING = False  # ligado o qwen3.8 gasta 5–16 mil tokens pensando e pode truncar

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    ROOT = Path("/content/projeto")
    if not (ROOT / "pipeline").is_dir():
        import zipfile
        from google.colab import files
        zipfile.ZipFile(next(iter(files.upload()))).extractall(ROOT)
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    !pip install -q paddleocr==3.7.0 paddlepaddle==3.3.1 "paddlex[ocr]==3.7.2" openai python-dotenv beautifulsoup4 opencv-contrib-python psutil
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import pandas as pd
from IPython.display import IFrame, display
from pipeline import metrics
from pipeline.ocr import load_engine, process_file
from pipeline.render import output_path, write_html

image, output = ROOT / IMAGE, ROOT / OUTPUT
show = lambda prefix: display(pd.DataFrame([r for r in metrics.summary() if r["stage"].startswith(prefix)]).round(2))

## 1. OCR

In [ ]:
with metrics.stage("ocr:carregar_modelo"):
    engine = load_engine(cpu_threads=CPU_THREADS)
folder, status = process_file(engine, image, image.parent, output)
layout = next(folder.glob("*_layout.json"))
print(status, layout)
show("ocr")

## 2. HTML via OpenRouter

In [ ]:
logs = [write_html(layout, kind, output, engine="openrouter", model=MODEL,
                   reasoning=REASONING) for kind in KINDS]
display(pd.DataFrame(logs))
show("html")

In [ ]:
IFrame(os.path.relpath(output_path(layout, KINDS[0], output)), width="100%", height=600)

## 3. Resumo

In [ ]:
metrics.report("notebook");

In [ ]:
if "google.colab" in sys.modules:  # baixa output/ e runs/ num zip
    !cd {ROOT} && zip -qr /content/colab_resultados.zip output runs
    from google.colab import files
    files.download("/content/colab_resultados.zip")